In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename)) 

In [2]:
# Install dependencies
!pip install -q transformers datasets peft bitsandbytes accelerate
!pip install -q huggingface_hub pandas numpy scikit-learn tqdm 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.0 MB/s eta 0:00:00:00:0100:01


In [3]:
import os, re, math, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from huggingface_hub import HfApi

warnings.filterwarnings('ignore')

SEED       = 42
MAX_LEN    = 256
BATCH_SIZE = 16
GRAD_ACCUM = 4
EPOCHS     = 5
LR         = 2e-4
MODEL_NAME = 'microsoft/deberta-v3-small' 
HUB_MODEL_ID = 'Idowenst/ecommerce-price-predictor-v1' #'Idowenst/jumia_dataset'  
OUTPUT_DIR = './price_predictor_output'

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [4]:
# Load dataset from HuggingFace
df = pd.read_parquet(
    'hf://datasets/Idowenst/jumia_dataset/data/train-00000-of-00001.parquet'
)
print('Shape:', df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head(3) 

Shape: (18360, 11)
price          float64
category        object
name            object
description     object
rating         float64
product_url     object
discount        object
old_price      float64
stock_info      object
num_reviews    float64
seller          object
dtype: object
price             0
category          0
name              0
description       0
rating         7767
product_url       0
discount          0
old_price      6872
stock_info        0
num_reviews    9951
seller          266
dtype: int64


,price,category,name,description,rating,product_url,discount,old_price,stock_info,num_reviews,seller
0,31700.0,appliances,SILVER CREST 2 Litres German Industrial Blende...,This Silver Crest 8500W Commercial Grinder ble...,3.9,https://www.jumia.com.ng/silver-crest-2-litres...,30%,45000.0,In stock,3304.0,Mummy's choices store
1,19330.0,appliances,Master Chef 3 In 1 Electric Blender With Mill ...,3 in 1 electric Blender with 2 mills It\'s so ...,3.8,https://www.jumia.com.ng/master-chef-3-in-1-el...,No discount,NaN,In stock,402.0,Glow-way forward Nig - AC
2,31750.0,appliances,Binatone 1.5 Litres (BLG-415) Blender With Gri...,This elegantly designed Binatone blender is eq...,4.0,https://www.jumia.com.ng/binatone-1.5-litres-b...,14%,36830.0,In stock,283.0,Jumia


In [5]:
# Auto-detect target (price) and feature columns 
def detect_price_column(df):
    candidates = [c for c in df.columns if any(
        kw in c.lower() for kw in ['price', 'cost', 'amount', 'value']
    )]
    if not candidates:
        num_cols = df.select_dtypes(include='number').columns.tolist()
        candidates = sorted(num_cols, key=lambda c: df[c].std(), reverse=True)
    return candidates[0]

def detect_text_columns(df):
    str_cols = df.select_dtypes(include='object').columns.tolist()
    return [c for c in str_cols
            if df[c].apply(lambda x: len(str(x)) if pd.notna(x) else 0).mean() > 10]

TARGET_COL = detect_price_column(df)
TEXT_COLS  = detect_text_columns(df)
NUM_COLS   = [c for c in df.select_dtypes(include='number').columns if c != TARGET_COL]

print(f'Target   : {TARGET_COL}')
print(f'Text cols: {TEXT_COLS}')
print(f'Num cols : {NUM_COLS}') 

Target   : price
Text cols: ['category', 'name', 'description', 'product_url', 'seller']
Num cols : ['rating', 'old_price', 'num_reviews']


In [6]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[^\w\s.,!?-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()[:512]

def parse_price(val):
    if isinstance(val, str):
        val = re.sub(r'[^\d.]', '', val)
        try: return float(val)
        except ValueError: return np.nan
    try: return float(val)
    except: return np.nan

df[TARGET_COL] = df[TARGET_COL].apply(parse_price)
df.dropna(subset=[TARGET_COL], inplace=True)
df.drop_duplicates(inplace=True)

# Remove price outliers
lo, hi = df[TARGET_COL].quantile([0.01, 0.99])
df = df[(df[TARGET_COL] >= lo) & (df[TARGET_COL] <= hi)].copy()

# Log-transform stabilises regression
df['log_price'] = np.log1p(df[TARGET_COL])

for col in TEXT_COLS:
    df[col] = df[col].apply(clean_text)

for col in NUM_COLS:
    df[col].fillna(df[col].median(), inplace=True)

# Composite text input
df['input_text'] = df[TEXT_COLS].apply(
    lambda r: ' | '.join(f'{c}: {v}' for c, v in r.items() if v), axis=1
)

print(f'Clean shape: {df.shape}')
print(df['input_text'].iloc[0]) 

Clean shape: (17174, 13)
category: appliances | name: SILVER CREST 2 Litres German Industrial Blender 8500W | description: This Silver Crest 8500W Commercial Grinder blender With Extra Mill Jar is suitable for hard foods and crush even ice block. it is specially made for food smoothening and has the ability to crush the seed of tomatoes to fine paste. it can be used for nuts, beans, maize, tomatoes, pepper,It s so important to have the right home appliances, whether they are large or small, they help you satisfy the different needs we have around the house as well as making life so much more comfortable for us. Here on Jumia, w | product_url: https www.jumia.com.ng silver-crest-2-litres-german-industrial-blender-8500w-72172768.html | seller: Mummy s choices store


In [7]:
train_df, val_df = train_test_split(
    df[['input_text', 'log_price']], test_size=0.1, random_state=SEED
)
print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}') 

Train: 15,456  |  Val: 1,718


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) 

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [9]:
class PriceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.tolist()
        self.labels    = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float),
        }

train_ds = PriceDataset(train_df['input_text'], train_df['log_price'], tokenizer, MAX_LEN)
val_ds   = PriceDataset(val_df['input_text'],   val_df['log_price'],   tokenizer, MAX_LEN) 

In [10]:
class PriceRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=8, lora_alpha=16, lora_dropout=0.1,
            target_modules=['query_proj', 'key_proj', 'value_proj'],
        )
        self.encoder = get_peft_model(self.encoder, lora_cfg)
        self.encoder.print_trainable_parameters()

        hidden = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Linear(256, 1),
        )

    def forward(self, input_ids, attention_mask, labels=None):
        out  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pool = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pred = self.regressor(pool).squeeze(-1)
        loss = nn.MSELoss()(pred, labels) if labels is not None else None
        return (loss, pred) if loss is not None else pred

model = PriceRegressor(MODEL_NAME).to(device)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

trainable params: 221,184 || all params: 141,525,504 || trainable%: 0.1563


In [11]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds_orig  = np.expm1(preds.flatten())
    labels_orig = np.expm1(labels)
    rmse = math.sqrt(mean_squared_error(labels_orig, preds_orig))
    mae  = mean_absolute_error(labels_orig, preds_orig)
    return {'rmse': rmse, 'mae': mae} 

In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rmse',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to='none',
    seed=SEED,
) 

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train() 

Epoch,Training Loss,Validation Loss,Rmse,Mae
1,2.796601,1.204271,161595.171413,61575.414062
2,1.033690,0.749034,147583.646140,55477.917969
3,0.736320,1.125013,199453.440702,91864.679688
4,0.654205,0.713892,178124.343087,67328.695312


TrainOutput(global_step=484, training_loss=9.195180266356665, metrics={'train_runtime': 481.1922, 'train_samples_per_second': 160.601, 'train_steps_per_second': 1.257, 'total_flos': 0.0, 'train_loss': 9.195180266356665, 'epoch': 4.0})

In [14]:
results = trainer.evaluate()
print(f"Val RMSE : {results['eval_rmse']:,.2f}")
print(f"Val MAE  : {results['eval_mae']:,.2f}") 

Val RMSE : 147,583.65
Val MAE  : 55,477.92


In [15]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.encoder.save_pretrained(f'{OUTPUT_DIR}/lora_encoder')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/tokenizer')
torch.save(model.regressor.state_dict(), f'{OUTPUT_DIR}/regressor_head.pt')
print('Model saved locally.') 

Model saved locally.


In [16]:
def predict_price(product_text, model, tokenizer, device):
    model.eval()
    enc = tokenizer(
        product_text, max_length=MAX_LEN,
        padding='max_length', truncation=True, return_tensors='pt',
    )
    with torch.no_grad():
        log_pred = model(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device),
        )
    return float(np.expm1(log_pred.cpu().numpy()))

sample = 'Brand: Samsung | Category: Electronics | Title: Galaxy A54 128GB | Description: 5G smartphone dual camera'
price  = predict_price(sample, model, tokenizer, device)
print(f'Predicted price: \u20a6{price:,.2f}') 

Predicted price: ₦444,980.00


In [17]:
HF_USERNAME = "Idowenst"

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN") 
login(token=hf_token) 

In [18]:
api = HfApi()
api.create_repo(repo_id=HUB_MODEL_ID, exist_ok=True)

model.encoder.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)
api.upload_file(
    path_or_fileobj=f'{OUTPUT_DIR}/regressor_head.pt',
    path_in_repo='regressor_head.pt',
    repo_id=HUB_MODEL_ID,
)
print(f'Pushed → https://huggingface.co/{HUB_MODEL_ID}') 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed → https://huggingface.co/Idowenst/ecommerce-price-predictor-v1
